In [0]:
%pip install -U optuna scikit-learn pandas numpy
dbutils.library.restartPython()


In [0]:
import pandas as pd
import numpy as np
from pyspark.sql.functions import col

train_path = "/Volumes/barbara_lakehouse/ml_sandbox/data/train.csv"
train_sdf = spark.read.csv(train_path, header=True, inferSchema=True)

# On prend un subset de colonnes typique (adapte si besoin)
cols = ["Transported", "HomePlanet", "Destination", "VIP", "CryoSleep",
        "Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

train_pd = train_sdf.select(*cols).toPandas()

# Normalize bools possibles ("True"/"False")
for c in ["VIP", "CryoSleep", "Transported"]:
    if train_pd[c].dtype == object:
        train_pd[c] = train_pd[c].astype(str).str.lower().map({"true": 1, "false": 0})
train_pd["Transported"] = train_pd["Transported"].astype(float)

In [0]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numerical_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
categorical_cols = ['HomePlanet', 'Destination', 'VIP', 'CryoSleep']

def objective(trial):
    df = train_pd.copy()

    # Split fixe (reproductible)
    X = df.drop(columns=["Transported"])
    y = df["Transported"].astype(int)

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    num_tf = Pipeline([("imputer", SimpleImputer(strategy="mean"))])
    cat_tf = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ])

    pre = ColumnTransformer(
        [("num", num_tf, numerical_cols),
         ("cat", cat_tf, categorical_cols)],
        remainder="drop"
    )

    # Hyperparams Optuna
    n_estimators = trial.suggest_int("n_estimators", 100, 600)
    max_depth = trial.suggest_int("max_depth", 3, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 50)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 30)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        n_jobs=-1,
        random_state=42
    )

    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_valid)

    return f1_score(y_valid, pred)

In [0]:
import optuna

study_name = "spaceship_optuna_rf"
storage = "sqlite:////Volumes/barbara_lakehouse/ml_sandbox/data/optuna_spaceship.db"  # UC volume path for serverless

def run_worker(worker_id: int, trials_per_worker: int):
    # IMPORTANT: load_if_exists=True pour que tous les workers rejoignent le même study
    study = optuna.create_study(
        study_name=study_name,
        storage=storage,
        direction="maximize",
        load_if_exists=True
    )
    study.optimize(objective, n_trials=trials_per_worker, n_jobs=1)

In [0]:
import optuna

# Run Optuna on the driver with parallel trials (works in serverless)
study = optuna.create_study(
    study_name="spaceship_optuna_rf",
    direction="maximize"
)

# n_jobs=-1 uses all available CPU cores on the driver for parallel trials
study.optimize(objective, n_trials=200, n_jobs=-1)

print("Best F1:", study.best_value)
print("Best params:", study.best_params)
print("Total trials:", len(study.trials))

In [0]:
study = optuna.load_study(study_name=study_name, storage=storage)
print("Best F1:", study.best_value)
print("Best params:", study.best_params)
print("Total trials:", len(study.trials))


In [0]:
storage = optuna.storages.RDBStorage(
    "sqlite:////dbfs/tmp/optuna_spaceship.db",
    engine_kwargs={"connect_args": {"timeout": 60}}
)
